# Support Vector Machines: A Detailed Mathematical Explanation

## 1. Basic Concept
Support Vector Machines (SVMs) are supervised learning models used for classification and regression analysis. The primary objective is to find a hyperplane that best separates different classes in the feature space (Cortes & Vapnik, 1995).

## 2. Linear SVM for Binary Classification
Given:
- Training data: $\{(x_1, y_1), (x_2, y_2), ..., (x_n, y_n)\}$
- $x_i \in \mathbb{R}^d$ (d-dimensional feature vectors)
- $y_i \in \{-1, +1\}$ (binary class labels)

The goal is to find a hyperplane:
\[ w \cdot x - b = 0 \]

Where:
- $w$ is the normal vector to the hyperplane
- $b$ is the bias term (Vapnik, 1998)

## 3. Optimal Hyperplane
The optimal hyperplane maximizes the margin between the two classes. The margin is the distance between the hyperplane and the nearest data point from either class.

For any point $x_i$:
$ y_i(w \cdot x_i - b) \geq 1 \quad \text{for all } i $

The width of the margin is $\frac{2}{||w||}$ (Bishop, 2006).

## 4. Optimization Problem
To find the optimal hyperplane, we need to solve:

minimize:
$ \frac{1}{2}||w||^2 $

subject to:
$ y_i(w \cdot x_i - b) \geq 1 \quad \text{for all } i $

This is a quadratic programming problem (Nello & Vapnik, 1995).

## 5. Lagrangian Formulation
We can use Lagrange multipliers to solve this:

$ L(w, b, \alpha) = \frac{1}{2}||w||^2 - \sum_i \alpha_i[y_i(w \cdot x_i - b) - 1] $

Where $\alpha_i$ are Lagrange multipliers (Pang, 2009).

## 6. Dual Problem
The dual problem is:

maximize:
$ \sum_i \alpha_i - \frac{1}{2}\sum_i \sum_j \alpha_i\alpha_jy_iy_j(x_i \cdot x_j) $

subject to:
$ \alpha_i \geq 0 \quad \text{for all } i, \quad \text{and} \quad \sum_i \alpha_iy_i = 0 $ (Schölkopf et al., 2001).

## 7. Support Vectors
The data points with $\alpha_i > 0$ are called support vectors. They are the points closest to the decision boundary (Vapnik, 1998).

## 8. Decision Function
For a new point $x$, the decision function is:

$ f(x) = \text{sign}\left(\sum_i \alpha_iy_i(x \cdot x_i) - b\right) $ (Cortes & Vapnik, 1995).

## 9. Kernel Trick
For non-linearly separable data, we can use the kernel trick. We replace the dot product $(x \cdot x_j)$ with a kernel function $K(x, x_j)$.

Common kernels include:
- Polynomial: $K(x, x_j) = (x \cdot x_j + c)^d$ (Bishop, 2006)
- Radial Basis Function (RBF): $K(x, x_j) = \exp(-\gamma||x - x_j||^2)$ (Schölkopf et al., 2001)

## 10. Soft Margin SVM
To handle outliers and overlapping classes, we introduce slack variables $\xi_i \geq 0$:

minimize:
\[ \frac{1}{2}||w||^2 + C \sum_i \xi_i \]

subject to:
\[ y_i(w \cdot x_i - b) \geq 1 - \xi_i \quad \text{and} \quad \xi_i \geq 0 \quad \text{for all } i \]

$C$ is a hyperparameter that controls the trade-off between maximizing the margin and minimizing the classification error (Cortes & Vapnik, 1995).

## Sequential Minimal Optimization (SMO) Algorithm
The SMO algorithm is used to solve the quadratic programming problem for SVMs efficiently (Platt, 1998).

### Pseudo Code for SMO Algorithm
```python
# Input: Training data {(x1, y1), (x2, y2), ..., (xn, yn)}
#        Kernel function K(xi, xj)
#        Regularization parameter C
# Output: Lagrange multipliers α and bias term b

def SMO(X, y, C, tol, max_passes):
    n = len(X)
    alpha = np.zeros(n)
    b = 0
    passes = 0

    while passes < max_passes:
        num_changed_alphas = 0
        for i in range(n):
            Ei = f(X[i]) - y[i]
            if (y[i]*Ei < -tol and alpha[i] < C) or (y[i]*Ei > tol and alpha[i] > 0):
                j = select_j(i, n)  # Randomly select j ≠ i
                Ej = f(X[j]) - y[j]

                alpha_i_old = alpha[i]
                alpha_j_old = alpha[j]

                if y[i] != y[j]:
                    L = max(0, alpha[j] - alpha[i])
                    H = min(C, C + alpha[j] - alpha[i])
                else:
                    L = max(0, alpha[i] + alpha[j] - C)
                    H = min(C, alpha[i] + alpha[j])

                if L == H:
                    continue

                eta = 2 * K(X[i], X[j]) - K(X[i], X[i]) - K(X[j], X[j])
                if eta >= 0:
                    continue

                alpha[j] -= y[j] * (Ei - Ej) / eta
                alpha[j] = clip(alpha[j], L, H)

                if abs(alpha[j] - alpha_j_old) < 1e-5:
                    continue

                alpha[i] += y[i] * y[j] * (alpha_j_old - alpha[j])

                b1 = b - Ei - y[i] * (alpha[i] - alpha_i_old) * K(X[i], X[i]) - y[j] * (alpha[j] - alpha_j_old) * K(X[i], X[j])
                b2 = b - Ej - y[i] * (alpha[i] - alpha_i_old) * K(X[i], X[j]) - y[j] * (alpha[j] - alpha_j_old) * K(X[j], X[j])

                if 0 < alpha[i] < C:
                    b = b1
                elif 0 < alpha[j] < C:
                    b = b2
                else:
                    b = (b1 + b2) / 2

                num_changed_alphas += 1

        if num_changed_alphas == 0:
            passes += 1
        else:
            passes = 0

    return alpha, b

```


## References

- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer.
- Cortes, C., & Vapnik, V. (1995). Support-Vector Networks. *Machine Learning*, 20(3), 273-297.
- Nello, G., & Vapnik, V. (1995). *The Nature of Statistical Learning Theory*. Springer.
- Pang, J. S. (2009). *Introduction to Optimization*. Springer.
- Platt, J. C. (1998). Sequential Minimal Optimization: A Fast Algorithm for Training Support Vector Machines. In *Advances in Kernel Methods: Support Vector Learning*. MIT Press.
- Schölkopf, B., Smola, A. J., & Müller, K. R. (2001). Nonlinear Support Vector Machines: A Review. In *Kernel Methods for Pattern Analysis*. Cambridge University Press.
- Vapnik, V. (1998). *Statistical Learning Theory*. Wiley.


In [ ]:
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Iris dataset
iris = datasets.load_iris()
X = iris.data[:, :2]  # We only take the first two features for simplicity
y = iris.target

# We only consider two classes for binary classification (e.g., class 0 and class 1)
X = X[y != 2]
y = y[y != 2]
y = np.where(y == 0, -1, 1)  # Convert class labels to -1 and 1

# Standardize the dataset
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Define helper functions for the SMO algorithm
def linear_kernel(x1, x2):
    return np.dot(x1, x2.T)

def compute_error(X, y, alpha, b, kernel, i):
    return np.dot((alpha * y), kernel(X, X[i])) + b - y[i]

def clip_alpha(alpha, L, H):
    return np.clip(alpha, L, H)

# Implement the SMO algorithm
def smo(X, y, C, tol, max_passes, kernel=linear_kernel):
    m, n = X.shape
    alpha = np.zeros(m)
    b = 0
    passes = 0

    while passes < max_passes:
        num_changed_alphas = 0
        for i in range(m):
            E_i = compute_error(X, y, alpha, b, kernel, i)

            if (y[i] * E_i < -tol and alpha[i] < C) or (y[i] * E_i > tol and alpha[i] > 0):
                j = np.random.randint(0, m)
                while j == i:
                    j = np.random.randint(0, m)

                E_j = compute_error(X, y, alpha, b, kernel, j)

                alpha_i_old = alpha[i].copy()
                alpha_j_old = alpha[j].copy()

                if y[i] == y[j]:
                    L = max(0, alpha[j] + alpha[i] - C)
                    H = min(C, alpha[j] + alpha[i])
                else:
                    L = max(0, alpha[j] - alpha[i])
                    H = min(C, C + alpha[j] - alpha[i])

                if L == H:
                    continue

                eta = 2.0 * kernel(X[i], X[j]) - kernel(X[i], X[i]) - kernel(X[j], X[j])
                if eta >= 0:
                    continue

                alpha[j] -= y[j] * (E_i - E_j) / eta
                alpha[j] = clip_alpha(alpha[j], L, H)

                if abs(alpha[j] - alpha_j_old) < tol:
                    alpha[j] = alpha_j_old
                    continue

                alpha[i] += y[i] * y[j] * (alpha_j_old - alpha[j])

                b1 = b - E_i - y[i] * (alpha[i] - alpha_i_old) * kernel(X[i], X[i]) - y[j] * (alpha[j] - alpha_j_old) * kernel(X[i], X[j])
                b2 = b - E_j - y[i] * (alpha[i] - alpha_i_old) * kernel(X[i], X[j]) - y[j] * (alpha[j] - alpha_j_old) * kernel(X[j], X[j])

                if 0 < alpha[i] < C:
                    b = b1
                elif 0 < alpha[j] < C:
                    b = b2
                else:
                    b = (b1 + b2) / 2

                num_changed_alphas += 1

        if num_changed_alphas == 0:
            passes += 1
        else:
            passes = 0

    return alpha, b

# Train the SVM using the SMO algorithm
C = 1.0
tol = 1e-3
max_passes = 5
alpha, b = smo(X_train, y_train, C, tol, max_passes)

# Helper function to compute the kernel matrix for predictions
def kernel_matrix(X1, X2, kernel=linear_kernel):
    return kernel(X1, X2)

# Make predictions
def predict(X, alpha, b, X_train, y_train, kernel=linear_kernel):
    K = kernel_matrix(X, X_train, kernel)
    return np.sign(np.dot(K, alpha * y_train) + b)

# Evaluate the SVM
y_pred = predict(X_test, alpha, b, X_train, y_train)
accuracy = np.mean(y_pred == y_test)
print(f"Accuracy: {accuracy * 100:.2f}%")


Accuracy: 100.00%
